In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
!rm -rf orbital-injury-detection/
!git clone https://github.com/Nadiam75/orbital-injury-detection.git

In [ ]:

import os

shared_base_directory = "/content/drive/MyDrive/Farabi"
import itertools
import random
import shutil
import gc
import cv2
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from prettytable import PrettyTable
from scipy import ndimage
from skimage import morphology
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import seaborn as sn # Often imported as sns, but sn is fine if consistent
import sys
repo_dir = os.path.join(os.getcwd(), "orbital-injury-detection")
sys.path.insert(0, repo_dir)

# from shallowC3D2 import *
from axCorCNNModel import *
from axCNNModel import *
from dicomdataset import *
from matplotlib import cm # Used by overlay_heatmap for colormap

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from utils import crop_image, normalize_image , apply_windowing , count_model_parameters

sys.path.append(shared_base_directory)



In [ ]:

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import cv2  # only used to resize CAMs

from utils import (
    add_img_subplot as _add_img_subplot,
    add_overlay_subplot as _add_overlay_subplot,
    ensure_same_hw as _ensure_same_hw,
    get_module_by_name,
    # normalize01 as _normalize01,
    pick_slice as _pick_slice,
)

def _normalize01(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    mn, mx = float(a.min()), float(a.max())
    if mx > mn:
        return (a - mn) / (mx - mn)
    return np.zeros_like(a, dtype=np.float32)


In [ ]:


output_train_axial_dir = os.path.join(shared_base_directory, "train-axial")
output_test_axial_dir = os.path.join(shared_base_directory, "test-axial")
output_train_coronal_dir = os.path.join(shared_base_directory, "train-coronal")
output_test_coronal_dir = os.path.join(shared_base_directory, "test-coronal")


for _d in (output_train_axial_dir, output_test_axial_dir, output_train_coronal_dir, output_test_coronal_dir):
    shutil.rmtree(_d, ignore_errors=True)


featuremaps_dir = "featuremaps"
gradcam_fracture_dir = "gradcam_fracture"  # Directory for Grad-CAM outputs

os.makedirs(output_train_axial_dir, exist_ok=True)
os.makedirs(output_test_axial_dir, exist_ok=True)
os.makedirs(output_train_coronal_dir, exist_ok=True)
os.makedirs(output_test_coronal_dir, exist_ok=True)

os.makedirs(featuremaps_dir, exist_ok=True)
os.makedirs(gradcam_fracture_dir, exist_ok=True)


# for the second model
# !rm -rf train_set/
# !rm -rf test_set/


In [ ]:


def plot_training_history(train_history: dict):
    """
    Plots the training and validation loss and accuracy over epochs.

    Args:
        train_history (dict): Dictionary containing 'train_loss', 'val_loss',
                              'train_accuracy', 'val_accuracy', and 'train_loss_without'.
    """
    epochs = range(1, len(train_history['train_loss']) + 1)

    plt.figure(figsize=(15, 5))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_history['train_loss'], label='Training Loss (with Reg.)')
    plt.plot(epochs, train_history['val_loss'], label='Validation Loss')
    plt.plot(epochs, train_history['train_loss_without'], label='Training Loss (without Reg.)')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_history['train_accuracy'], label='Training Accuracy')
    plt.plot(epochs, train_history['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def train_model(model: nn.Module, trainloader: DataLoader, validationloader: DataLoader,
              num_epochs: int = 10, lr: float = 0.0001, l1_lambda: float = 0.0001,
              l2_lambda: float = 0.0001) -> nn.Module:
    """
    Trains a PyTorch model and tracks its performance.

    Args:
        model (nn.Module): The PyTorch model to train.
        trainloader (DataLoader): DataLoader for the training data.
        validationloader (DataLoader): DataLoader for the validation data.
        num_epochs (int): Number of training epochs. Defaults to 10.
        lr (float): Learning rate for the optimizer. Defaults to 0.0001.
        l1_lambda (float): L1 regularization strength. Defaults to 0.0001.
        l2_lambda (float): L2 regularization strength. Defaults to 0.0001.

    Returns:
        nn.Module: The best performing model (based on validation accuracy).
    """
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    train_history = {'train_loss': [], 'train_loss_without': [],
                     'train_accuracy': [], 'val_loss': [], 'val_accuracy': []}

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(params=model.parameters(), lr=lr)

    max_val_acc = -1.0
    best_model_wts = None
    best_epoch = -1

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_loss_without_reg = 0.0
        correct_train = 0
        total_train = 0

        for i, data in enumerate(trainloader, 0):
            ax_input = data['axial'].type(torch.cuda.FloatTensor).to(device)
            cor_input = data['coronal'].type(torch.cuda.FloatTensor).to(device)
            va_logmar_input = data['VAlogmar'].type(torch.cuda.FloatTensor).to(device)
            labels = data['label'].type(torch.cuda.FloatTensor).to(device)

            optimizer.zero_grad()

            outputs = model(ax_input, cor_input, va_logmar_input)
            loss_without_reg = criterion(outputs, labels)
            running_loss_without_reg += loss_without_reg.item()

            l1_norm = sum(p.abs().sum() for p in model.parameters() if p.requires_grad)
            l2_norm = sum(p.pow(2.0).sum() for p in model.parameters() if p.requires_grad)
            loss = loss_without_reg + l1_lambda * l1_norm + l2_lambda * l2_norm

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            predicted_train = F.sigmoid(outputs).round()
            total_train += len(labels)
            correct_train += (predicted_train == labels).sum().item()

        train_accuracy = (correct_train / total_train) * 100
        train_loss = running_loss / len(trainloader)
        train_loss_without_reg = running_loss_without_reg / len(trainloader)

        train_history['train_loss'].append(train_loss)
        train_history['train_loss_without'].append(train_loss_without_reg)
        train_history['train_accuracy'].append(train_accuracy)

        # Validation phase
        model.eval()
        correct_val = 0
        total_val = 0
        running_val_loss = 0.0
        with torch.no_grad():
            for data_val in validationloader:
                ax_input_val = data_val['axial'].type(torch.cuda.FloatTensor).to(device)
                cor_input_val = data_val['coronal'].type(torch.cuda.FloatTensor).to(device)
                va_logmar_input_val = data_val['VAlogmar'].type(torch.cuda.FloatTensor).to(device)
                labels_val = data_val['label'].type(torch.cuda.FloatTensor).to(device)

                outputs_val = model(ax_input_val, cor_input_val, va_logmar_input_val)
                loss_val = criterion(outputs_val, labels_val)
                running_val_loss += loss_val.item()

                predicted_val = F.sigmoid(outputs_val).round()
                total_val += len(labels_val)
                correct_val += (predicted_val == labels_val).sum().item()

        val_accuracy = (correct_val / total_val) * 100
        val_loss = running_val_loss / len(validationloader)

        train_history['val_loss'].append(val_loss)
        train_history['val_accuracy'].append(val_accuracy)

        print(
            f'Epoch:{epoch+1:2d}, '
            f'Train Loss: {train_loss:.4f} (No Reg: {train_loss_without_reg:.4f}), '
            f'Train Acc: {train_accuracy:.2f}% | '
            f'Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%'
        )

        if val_accuracy > max_val_acc:
            max_val_acc = val_accuracy
            best_model_wts = model.state_dict()
            best_epoch = epoch + 1
            torch.save(best_model_wts, 'best_model.pth')
            print(f"  --> Saved best model weights at epoch {best_epoch} with validation accuracy {max_val_acc:.2f}%")

    print(f'\nTraining finished. Best validation accuracy: {max_val_acc:.2f}% at epoch {best_epoch}.')

    model.load_state_dict(best_model_wts)
    return model.cpu() , train_history

def test_model(model: nn.Module, testloader: DataLoader) -> None:
    """
    Evaluates the trained model on the test dataset and plots the confusion matrix.

    Args:
        model (nn.Module): The trained PyTorch model.
        testloader (DataLoader): DataLoader for the test data.
    """
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.eval()
    model.to(device)

    correct_test = 0
    total_test = 0
    conf_matrix = torch.zeros(2, 2)

    with torch.no_grad():
        for data_test in testloader:
            ax_input = data_test['axial'].type(torch.cuda.FloatTensor).to(device)
            cor_input = data_test['coronal'].type(torch.cuda.FloatTensor).to(device)
            va_logmar_input = data_test['VAlogmar'].type(torch.cuda.FloatTensor).to(device)
            labels = data_test['label'].type(torch.cuda.FloatTensor).to(device)

            outputs = model(ax_input, cor_input, va_logmar_input)
            predicted = F.sigmoid(outputs).round()

            total_test += len(labels)
            correct_test += (predicted == labels).sum().item()

            for t, p in zip(labels.view(-1), predicted.view(-1)):\
                conf_matrix[t.long(), p.long()] += 1

    print(f'Accuracy on test images: {correct_test / total_test * 100:.2f}% ({correct_test}/{total_test})')
    # print('Generating confusion matrix...')
    return conf_matrix

# def plot_confusion_matrix(conf_matrix: torch.Tensor, show: bool = True):
    # """
    # Plots a confusion matrix using matplotlib.

    # Args:
    #     conf_matrix (torch.Tensor): The confusion matrix tensor (e.g., from test_model).
    #     show (bool): If True, displays the plot. Defaults to True.
    # """
    # cm = conf_matrix.cpu().numpy()
    # cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # plt.figure(figsize=(5, 5))
    # plt.imshow(cm_normalized, interpolation='nearest', cmap=plt.cm.Blues)
    # plt.title("Confusion Matrix (Normalized)")
    # plt.colorbar()

    # class_labels = ["No Fracture (0)", "Fracture (1)"]
    # tick_marks = np.arange(len(class_labels))
    # plt.xticks(tick_marks, class_labels, rotation=45, ha='right')
    # plt.yticks(tick_marks, class_labels)

    # fmt = '.2f'
    # thresh = cm_normalized.max() / 2.
    # for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    #     plt.text(j, i, format(cm_normalized[i, j], fmt),
    #              horizontalalignment="center",
    #              color="white" if cm_normalized[i, j] > thresh else "black")
    #     plt.text(j, i + 0.3, f'({int(cm[i,j])})',
    #              horizontalalignment="center",
    #              color="white" if cm_normalized[i, j] > thresh else "black",
    #              fontsize=8)

    # plt.tight_layout()
    # plt.savefig('confusion_matrix_fracture.png')

    # if show:
    #     plt.show()



In [ ]:

def overlay_heatmap(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """
    Overlays a heatmap on an image.

    Args:
        img (np.ndarray): The input image (H, W) or (H, W, C).
        heatmap (np.ndarray): The Grad-CAM heatmap (H', W'), typically normalized to [0, 1].
        alpha (float): Transparency of the heatmap (0.0 to 1.0). Defaults to 0.5.

    Returns:
        np.ndarray: The superimposed image (H, W, 3) in BGR format with heatmap overlay.
    """
    # Resize heatmap to match image dimensions
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))

    # Normalize heatmap to [0, 255] and apply colormap
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, cv2.COLORMAP_JET)

    # Convert image to BGR if it's grayscale (1 channel)
    if img.ndim == 2:
        img_bgr = cv2.cvtColor(np.uint8(img), cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 1: # (H, W, 1)
         img_bgr = cv2.cvtColor(np.uint8(img.squeeze()), cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 3: # (H, W, 3) already BGR or RGB, assume BGR for cv2
        img_bgr = np.uint8(img)
    else:
        print("Warning: Input image for overlay_heatmap might not be 2D or 3-channel.")
        # Attempt to use the first channel if more than 3, or convert to grayscale.
        if img.ndim == 3 and img.shape[2] > 3:
            img_bgr = cv2.cvtColor(np.uint8(img[:,:,0]), cv2.COLOR_GRAY2BGR) # Use first channel
        else:
            img_bgr = np.uint8(img)


    # Superimpose the heatmap on the image
    # Note: cv2.addWeighted uses img * alpha + heatmap * beta.
    # To make heatmap transparent and image prominent: image_weight=1-alpha, heatmap_weight=alpha
    superimposed_img = cv2.addWeighted(img_bgr, 1 - alpha, heatmap_colored, alpha, 0)
    return superimposed_img



In [ ]:
font_size = 36
def plot_confusion_matrix_plotly(conf_matrix: torch.Tensor, scale):
    """
    Plots a confusion matrix using Plotly.

    Args:
        conf_matrix (torch.Tensor): The confusion matrix tensor.
        show (bool): If True, displays the plot. Defaults to True.
    """
    cm = conf_matrix.cpu().numpy()
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    labels_y = ["No Fracture (0)", "Fracture (1)"]

    labels_x = ["No Fracture (0)", "Fracture (1)"]
    class_labels = ["No Fracture (0)", "Fracture (1)"]

    # Create text annotations (normalized + raw count)
    annotations = [[f"{cm_normalized[i, j]:.2f}%<br>({cm[i, j]})" for j in range(cm.shape[1])]
                   for i in range(cm.shape[0])]

    # Define custom peach colormap
    peach_colorscale = [
        [0, "#FFE5B4"],  # light peach
        [0.5, "#FFCC99"],  # medium peach
        [1, "#FF9966"]   # darker peach
    ]

    fig = go.Figure(data=go.Heatmap(
        z=cm_normalized,
        x=class_labels,
        y=class_labels,
        text=annotations,
        textfont={"size": 36, "color": "black"},  # annotations black

        texttemplate="%{text}",
        colorscale=peach_colorscale,
        showscale=True,
        hoverongaps=False,
        # colorbar=dict(title="Proportion")
    ))

    fig.update_layout(
        title=dict(
            # text="Orbital Fracture Detection",
            text="Fracture Detection Confusion Matrix",
            x=0.5,                # center title
            xanchor="center"
        ),
        xaxis_title="Predicted",
        yaxis_title="Actual",
        width=1300*scale, height=1040*scale,
        margin=dict(l=60, r=40, t=120, b=60),
        paper_bgcolor="white",   # white background outside plot
        plot_bgcolor="white" ,   # white background inside plot
            font=dict(size=font_size, )  # axis, ticks, colorbar font

    )

    fig.update_xaxes(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=labels_x
    )
    fig.update_yaxes(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=labels_y,
          # textfont={"size": 36, "color": "black"},  # annotations black

        tickangle=-90   # rotate y-axis ticks 90 degrees
    )




    return fig




def plot_training_history(train_history: dict):
    """
    Plots the training and validation loss and accuracy over epochs using Plotly
    with a white background and black text.
    """
    epochs = list(range(1, len(train_history['train_loss']) + 1))

    # Create subplots: 1 row, 2 columns
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("Training and Validation Loss", "Training and Validation Accuracy"))

    # --- Loss traces ---
    # fig.add_trace(go.Scatter(
    #     x=epochs, y=train_history['train_loss'],
    #     mode='lines+markers',
    #     name='Training Loss',
    #     line=dict(color="black")
    # ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=epochs,  y=train_history['train_loss_without'],
        mode='lines+markers',
        name='Validation Loss',
        line=dict(color="blue")
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=epochs,  y=train_history['val_loss'],

        mode='lines+markers',
        name='Training Loss',
        line=dict(color="black")
    ), row=1, col=1)

    # --- Accuracy traces ---
    fig.add_trace(go.Scatter(
        x=epochs, y=train_history['val_accuracy'],
        mode='lines+markers',
        name='Training Accuracy',
        line=dict(color="black")
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=epochs, y=train_history['train_accuracy'],
        mode='lines+markers',
        name='Validation Accuracy',
        line=dict(color="blue")
    ), row=1, col=2)

    # Update layout for white background + black text
    fig.update_layout(
        title_text="Training History",
        width=1200,
        height=500,
        legend=dict(x=0.5, y=-0.15, orientation="h", xanchor="center"),
        margin=dict(t=80, b=80),
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(color="black")
    )

    # Update axes style
    fig.update_xaxes(title_text="Epochs", showgrid=True, gridcolor="lightgray", zeroline=False)
    fig.update_yaxes(title_text="Loss", row=1, col=1, showgrid=True, gridcolor="lightgray", zeroline=False)
    fig.update_xaxes(title_text="Epochs", row=1, col=2, showgrid=True, gridcolor="lightgray", zeroline=False)
    fig.update_yaxes(title_text="Accuracy (%)", row=1, col=2, showgrid=True, gridcolor="lightgray", zeroline=False)

    fig.show()



def plot_image_slices_and_heatmap(cor , img, cor_heatmap, ax_heatmap, label, predicted):
    # Pick the middle slice (as you originally plotted img[i] for i in range(12, 18))
    # mid_slice_index =
    slice_img = img[15]
    slice_cor = cor [25]
     # heatmap = heatmap.T
    # slice_img = slice_img.T
    # Create subplots
    fig = make_subplots(rows=2, cols=2,

        horizontal_spacing=0.02,

        subplot_titles=("Coronal Slice", "Coronal Gradcam Heatmap" , "Axial Slice" , "Axial Gradcam Heatmap"))

    # Add mid slice image
    fig.add_trace(
        go.Heatmap(z=slice_cor, colorscale='gray', showscale=False),
        row=1, col=1
    )


    fig.add_trace(
        go.Heatmap(z=slice_img, colorscale='gray', showscale=False),
        row=2, col=1
    )

    # Add heatmap
    fig.add_trace(
        go.Heatmap(z=cor_heatmap, colorscale='Jet', showscale=True),
        row=1, col=2
    )


    fig.add_trace(
        go.Heatmap(z=ax_heatmap, colorscale='Jet', showscale=True),
        row=2, col=2
    )
    # Set the overall layout with custom font and title
    fig.update_layout(
        title={
            'text': f'True Label: {label} Injury - Predicted Label : {predicted} Injury',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'family': 'Times New Roman', 'size': 18}
        },
        font=dict(family="Times New Roman"),
        height=800,
        width=800,
        # margin=dict(t=60)
    )

    # Remove axis ticks and labels
    fig.update_xaxes(showticklabels=False).update_yaxes(showticklabels=False)

    return fig


In [9]:
# Load raw data
coronal_train_raw = np.load(os.path.join(shared_base_directory, "train-test+widerange", "train_coronal.npy"))
coronal_test_raw = np.load(os.path.join(shared_base_directory, "train-test+widerange", "test_coronal.npy"))
axial_train_raw = np.load(os.path.join(shared_base_directory, "train-test+widerange", "axial2_train200.npy"))
axial_test_raw = np.load(os.path.join(shared_base_directory, "train-test+widerange", "axial2_test50.npy"))

df_train_raw = pd.read_csv(os.path.join(shared_base_directory, "train-test+widerange", "Data_GT_train2.csv"))
df_test_raw = pd.read_csv(os.path.join(shared_base_directory, "train-test+widerange", "Data_GT_test2.csv"))


# --- Data Transformation (Concatenate and Flip) ---
# This part effectively doubles the dataset by mirroring each patient's data
# and assigning it to the other eye's perspective.
axial_train_transformed = np.concatenate([axial_train_raw[:,:,:120,:], np.flip(axial_train_raw[:,:,120:,:], axis=2)])
coronal_train_transformed = np.concatenate([coronal_train_raw[:,:,:256,:], np.flip(coronal_train_raw[:,:,256:,:], axis=2)])

axial_test_transformed = np.concatenate([axial_test_raw[:,:,:120,:], np.flip(axial_test_raw[:,:,120:,:], axis=2)])
coronal_test_transformed = np.concatenate([coronal_test_raw[:,:,:256,:], np.flip(coronal_test_raw[:,:,256:,:], axis=2)])


del coronal_train_raw , coronal_test_raw , axial_train_raw , axial_test_raw


print(f"Axial Train Transformed Shape: {axial_train_transformed.shape}")
print(f"Coronal Train Transformed Shape: {coronal_train_transformed.shape}")
print(f"Axial Test Transformed Shape: {axial_test_transformed.shape}")
print(f"Coronal Test Transformed Shape: {coronal_test_transformed.shape}")




Axial Train Transformed Shape: (400, 150, 120, 40)
Coronal Train Transformed Shape: (400, 256, 256, 32)
Axial Test Transformed Shape: (100, 150, 120, 40)
Coronal Test Transformed Shape: (100, 256, 256, 32)


In [10]:


# --- Labels and VAlogmar Preparation ---
def prepare_labels_and_valogmars(df: pd.DataFrame) -> tuple:
    """
    Prepares labels (OS/OD) and VAlogmars from the DataFrame.
    The data is duplicated to match the mirrored image data.
    """
    num_data = len(df)
    os_labels = np.zeros((num_data, 1), dtype=np.float32)
    od_labels = np.zeros((num_data, 1), dtype=np.float32)
    va_logmars = np.zeros((num_data, 1), dtype=np.float32)

    for i, patient_numpy in enumerate(df.to_numpy()):
        # Assuming df structure: code, ONSbinary, ONS, IOS, IOSbinary, IOSONS, OS_, OD_, VAlogmar
        _, _, _, _, _, _, os_val, od_val, va_logmar_val = patient_numpy
        va_logmars[i] = va_logmar_val
        os_labels[i] = int(os_val)
        od_labels[i] = int(od_val)

    # Concatenate labels and VAlogmars to match the doubled image data
    combined_labels = np.concatenate([os_labels, od_labels])
    combined_valogmars = np.concatenate([va_logmars, va_logmars])
    return combined_labels, combined_valogmars


labels_train, VAlogmars_train = prepare_labels_and_valogmars(df_train_raw)
labels_test, VAlogmars_test = prepare_labels_and_valogmars(df_test_raw)

print(f"\nCombined Labels Train Shape: {labels_train.shape}")
print(f"Combined VAlogmars Train Shape: {VAlogmars_train.shape}")
print(f"Combined Labels Test Shape: {labels_test.shape}")
print(f"Combined VAlogmars Test Shape: {VAlogmars_test.shape}")




Combined Labels Train Shape: (400, 1)
Combined VAlogmars Train Shape: (400, 1)
Combined Labels Test Shape: (100, 1)
Combined VAlogmars Test Shape: (100, 1)


In [11]:
# Initialize label/VAlogmar lists
labels_list = []
valogmars_list = []


def random_rotate(sample: np.ndarray, degrees: float = 15) -> np.ndarray:
    """Randomly rotates a 3D image slice-by-slice."""
    angle = random.uniform(-degrees, degrees)
    rotated_image = np.zeros_like(sample)
    for i in range(sample.shape[2]): # Iterate through depth slices
        M = cv2.getRotationMatrix2D((sample.shape[1] / 2, sample.shape[0] / 2), angle, 1)
        rotated_image[:,:,i] = cv2.warpAffine(sample[:,:,i], M,
                                               (sample.shape[1], sample.shape[0]), flags=cv2.INTER_LINEAR)
    return rotated_image


# Process and save augmented training data
num_train_samples = axial_train_transformed.shape[0]
counter = 0

for i in range(num_train_samples):
    ax_sample = axial_train_transformed[i]
    cor_sample = coronal_train_transformed[i]
    lbl = labels_train[i]
    va_logmar = VAlogmars_train[i]

    # Save original
    np.save(os.path.join(output_train_axial_dir, f'{counter}.npy'), ax_sample)
    np.save(os.path.join(output_train_coronal_dir, f'{counter}.npy'), cor_sample)
    labels_list.append(lbl)
    valogmars_list.append(va_logmar)
    counter += 1

    # Save rotated
    rotated_ax = random_rotate(ax_sample)
    rotated_cor = random_rotate(cor_sample)
    np.save(os.path.join(output_train_axial_dir, f'{counter}.npy'), rotated_ax)
    np.save(os.path.join(output_train_coronal_dir, f'{counter}.npy'), rotated_cor)
    labels_list.append(lbl)
    valogmars_list.append(va_logmar)
    counter += 1

    if i % 50 == 0:
        print(f"Processed {i}/{num_train_samples} samples...")
        gc.collect()

# Save test data (non-augmented)
for i, sample in enumerate(axial_test_transformed):
    np.save(os.path.join(output_test_axial_dir, f'{i}.npy'), sample)

for i, sample in enumerate(coronal_test_transformed):
    np.save(os.path.join(output_test_coronal_dir, f'{i}.npy'), sample)

# Save labels and VAlogmars
final_labels_train = np.array(labels_list).reshape(-1, 1)
final_valogmars_train = np.array(valogmars_list).reshape(-1, 1)
np.save("final_labels_train.npy", final_labels_train)
np.save("final_valogmars_train.npy", final_valogmars_train)

# Report
print(f"\nFinal Axial Train Samples Saved: {len(os.listdir(output_train_axial_dir))}")
print(f"Final Coronal Train Samples Saved: {len(os.listdir(output_train_coronal_dir))}")
print(f"Final Axial Test Samples Saved: {len(os.listdir(output_test_axial_dir))}")
print(f"Final Coronal Test Samples Saved: {len(os.listdir(output_test_coronal_dir))}")
print(f"Final labels_train shape: {final_labels_train.shape}")
print(f"Final VAlogmars_train shape: {final_valogmars_train.shape}")


Processed 0/400 samples...
Processed 50/400 samples...
Processed 100/400 samples...
Processed 150/400 samples...
Processed 200/400 samples...
Processed 250/400 samples...
Processed 300/400 samples...
Processed 350/400 samples...

Final Axial Train Samples Saved: 800
Final Coronal Train Samples Saved: 800
Final Axial Test Samples Saved: 100
Final Coronal Test Samples Saved: 100
Final labels_train shape: (800, 1)
Final VAlogmars_train shape: (800, 1)


In [12]:
num_data = len(df_train_raw)
OS = np.zeros((num_data,1), dtype=np.float32)
OD = np.zeros((num_data,1), dtype=np.float32)
VAlogmars = np.zeros((num_data,1), dtype=np.float32)
labels_train = df_train_raw.to_numpy()
for i,patient_numpy  in enumerate(labels_train):
    code, ONSbinary , ONS,IOS,IOSbinary , IOSONS,OS_, OD_, VAlogmar = patient_numpy
    VAlogmars[i] = VAlogmar
    OS[i] = int(OS_)
    OD[i] = int(OD_)

VAlogmars_train = np.concatenate([VAlogmars,VAlogmars])
labels_train = np.concatenate([OS,OD])
len(labels_train)

400

In [13]:
OD.shape

(200, 1)

In [14]:
num_data = len(df_test_raw)
OS = np.zeros((num_data,1), dtype=np.float32)
OD = np.zeros((num_data,1), dtype=np.float32)
VAlogmars = np.zeros((num_data,1), dtype=np.float32)
labels_test = df_test_raw.to_numpy()
for i,patient_numpy  in enumerate(labels_test):
    code, ONSbinary , ONS,IOS,IOSbinary , IOSONS,OS_, OD_, VAlogmar = patient_numpy
    VAlogmars[i] = VAlogmar
    OS[i] = int(OS_)
    OD[i] = int(OD_)

VAlogmars_test = np.concatenate([VAlogmars,VAlogmars])
labels_test = np.concatenate([OS,OD])
len(labels_test)

100

In [15]:
axial_train_transformed.shape

(400, 150, 120, 40)

In [16]:
coronal_train_transformed.shape

(400, 256, 256, 32)

In [17]:
#  UNCOMMENT IF YOU WANNA PLOT DATASET
# # 0 , 10, 15 , 45
# # --- Inputs you already have ---
# # axial_train_processed: np.ndarray of shape (N, H, W, C)
# # labels_train: torch.Tensor or np.ndarray of shape (N,)
# # offs = 45

# os.makedirs('axial_fracture_images')
# for offs in range(0 , 200 , 5):
#   # print(offs)
#   # indices_to_plot = [offs +25, offs +60, offs+120, 100]
#   indices_to_plot = [1+offs, 2+offs, 3+offs, 4 + offs]

#   # Robust label extraction (works for torch.Tensor or numpy)
#   def get_label_scalar(x):
#       try:
#           return int(x.item())
#       except AttributeError:
#           return int(x)

#   labels_for_plot = [get_label_scalar(labels_train[i]) for i in indices_to_plot]
#   label_strings = ['Fracture (1)' if l == 1 else 'No fracture (0)' for l in labels_for_plot]

#   # Grab the images (single axial slice per index)
#   imgs = [axial_train_transformed[i, :, :, 20] for i in indices_to_plot]

#   # --- Plotly figure ---
#   fig = make_subplots(
#       rows=2, cols=2,
#       subplot_titles=[f' {s}' for s in label_strings],
#       vertical_spacing=0.08, horizontal_spacing=0.05
#   )

#   for k, img in enumerate(imgs):
#       r = k // 2 + 1
#       c = k % 2 + 1
#       # Add as heatmap to get a grayscale colorscale
#       fig.add_trace(
#           go.Heatmap(
#               z=img,
#               colorscale='gray',   # grayscale like matplotlib cmap='gray'
#               showscale=False,
#               zsmooth=False
#           ),
#           row=r, col=c
#       )
#       # Hide axes for that subplot
#       fig.update_xaxes(visible=False, row=r, col=c)
#       fig.update_yaxes(visible=False, row=r, col=c)

#   fig.update_layout(
#       title_text='Sample Axial Images from Training Data',
#       title_x=0.5,
#       height=800, width=800,
#       margin=dict(l=10, r=10, t=60, b=10),
#   )

#   fig.show()

#   # Optional: save a static PNG (requires `pip install -U kaleido`)
#   fig.write_html(f'axial_fracture_images/sample_axial_image_{offs}.html',)

In [18]:
# UNCOMMENT IF YOU WANNA PLOT THE DATASET
# # 0 , 10, 15 , 45
# # --- Inputs you already have ---
# # axial_train_processed: np.ndarray of shape (N, H, W, C)
# # labels_train: torch.Tensor or np.ndarray of shape (N,)
# # offs = 45
# !rm -rf coronal_fracture_images
# os.makedirs('coronal_fracture_images')
# for offs in range(0 , 200 , 5):
#   # print(offs)
#   # indices_to_plot = [offs +25, offs +60, offs+120, 100]
#   indices_to_plot = [1+offs, 2+offs, 3+offs, 4 + offs]

#   # Robust label extraction (works for torch.Tensor or numpy)
#   def get_label_scalar(x):
#       try:
#           return int(x.item())
#       except AttributeError:
#           return int(x)

#   labels_for_plot = [get_label_scalar(labels_train[i]) for i in indices_to_plot]
#   label_strings = ['Fracture (1)' if l == 1 else 'No fracture (0)' for l in labels_for_plot]

#   # Grab the images (single axial slice per index)
#   imgs = [coronal_train_transformed[i, :, :, 16] for i in indices_to_plot]

#   # --- Plotly figure ---
#   fig = make_subplots(
#       rows=2, cols=2,
#       subplot_titles=[f' {s}' for s in label_strings],
#       vertical_spacing=0.08, horizontal_spacing=0.05
#   )

#   for k, img in enumerate(imgs):
#       r = k // 2 + 1
#       c = k % 2 + 1
#       # Add as heatmap to get a grayscale colorscale
#       fig.add_trace(
#           go.Heatmap(
#               z=img,
#               colorscale='gray',   # grayscale like matplotlib cmap='gray'
#               showscale=False,
#               zsmooth=False
#           ),
#           row=r, col=c
#       )
#       # Hide axes for that subplot
#       fig.update_xaxes(visible=False, row=r, col=c)
#       fig.update_yaxes(visible=False, row=r, col=c)

#   fig.update_layout(
#       title_text='Sample Coronal Images from Training Data',
#       title_x=0.5,
#       height=800, width=800,
#       margin=dict(l=10, r=10, t=60, b=10),
#   )

#   fig.show()

#   # Optional: save a static PNG (requires `pip install -U kaleido`)
#   fig.write_html(f'coronal_fracture_images/sample_coronal_image_{offs}.html',)

In [19]:
# !zip -r axial_fracture_images.zip axial_fracture_images

In [20]:
# !zip -r coronal_fracture_images.zip coronal_fracture_images

In [21]:
for i,sample in enumerate(coronal_train_transformed):
    np.save(os.path.join(shared_base_directory, "train-coronal", f"{i}.npy"), sample)
for i,sample in enumerate(axial_train_transformed):
    np.save(os.path.join(shared_base_directory, "train-axial", f"{i}.npy"), sample)
for i,sample in enumerate(axial_test_transformed):
    np.save(os.path.join(shared_base_directory, "test-axial", f"{i}.npy"), sample)
for i,sample in enumerate(coronal_test_transformed):
    np.save(os.path.join(shared_base_directory, "test-coronal", f"{i}.npy"), sample)



In [22]:
num_data

50

In [23]:
len_axial = axial_train_transformed.shape[0]
new_index = 0

new_labels = []
new_good_samples_min_max = []
num_data = sum(1 for f in os.listdir(os.path.join(shared_base_directory, "train-axial")) if f.endswith('.npy'))



In [24]:
# num_train_samples = 400
# for i in range(  num_train_samples):


#     sample_ax = np.load(os.path.join(shared_base_directory, "train-axial", f"{i}.npy"))
#     res_ax = sample_ax * 1
#     res_ax = RandomRotate(res_ax)

#     sample_cor = np.load(os.path.join(shared_base_directory, "train-coronal", f"{i}.npy"))
#     res_cor = sample_cor * 1
#     res_cor = RandomRotate(res_cor)
#     lbl = labels_train[i]


#     new_labels.append(lbl)
#     idx = i + num_data
#     np.save(os.path.join(shared_base_directory, "train-axial", f"{idx}.npy"), res_ax)

#     np.save(os.path.join(shared_base_directory, "train-coronal", f"{idx}.npy"), res_cor)

#     if i % 50 == 0:
#         print(f"Processed {i}/{num_train_samples} samples...")
#         gc.collect()

#     '''small_array_expanded = np.expand_dims(res, axis=0)
#     axial = np.concatenate((axial, small_array_expanded), axis=0)
#     label_IOS = np.append(label_IOS, lbl)

#     small_array_expanded = np.expand_dims(test, axis=0)
#     good_samples_min_max = np.concatenate((good_samples_min_max, small_array_expanded), axis=0)'''


# new_labels = np.array(new_labels)
# labels_train = np.concatenate((labels_train,new_labels),axis=0)
# print(len(os.listdir(os.path.join(shared_base_directory, "train-axial"))), labels_train.shape)
# VAlogmars_train = np.concatenate([VAlogmars_train,VAlogmars_train])



In [25]:
np.save(os.path.join(shared_base_directory, "new_train_labels.npy"), labels_train)



In [26]:
# Main execution block
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
batch_size = 8

# Define image transforms
image_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Create Dataset and DataLoader for training
trainset = DICOMDataset(data_dir_prefix='train',
                        # shared_base_directory=shared_base_directory,
                        VAlogmars=final_valogmars_train,
                        labels=final_labels_train,
                        transform=image_transform)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

# Create Dataset and DataLoader for validation/testing
validationset = DICOMDataset(data_dir_prefix='test',
                            #  shared_base_directory=shared_base_directory,
                             VAlogmars=VAlogmars_test,
                             labels=labels_test,
                             transform=image_transform)
validationloader = DataLoader(validationset, batch_size=batch_size, shuffle=True, num_workers=2)

print(f"Number of batches in trainloader: {len(trainloader)}")
print(f"Number of batches in validationloader: {len(validationloader)}")


# Initialize and train the model
model = AxCorCNN_Model(multiclass=False, mode='ax')
# print("\nModel Parameters before training:")
total_params = count_model_parameters(model)
# print(f'Total number of parameters in the model: {total_params}')

trained_model , train_history = train_model(model, trainloader, validationloader,
                          num_epochs=20, lr=1e-4, l1_lambda=1e-4, l2_lambda=1e-4)

# Load the best model weights for final testing
final_model_for_test = AxCorCNN_Model(multiclass=False, mode='ax')
final_model_for_test.load_state_dict(torch.load('best_model.pth'))
final_model_for_test.eval()

# Test the model
conf_matrix = test_model(final_model_for_test, validationloader)


# --- Example of checking DataLoader output structure (uncomment to run) ---
# for item in trainloader:
#   print(item.keys())
#   print(item['axial'].shape)
#   print(item['coronal'].shape)
#   print(item['VAlogmar'].shape)
#   print(item['label'].shape)
#   break

# --- Example of checking layer output shapes (uncomment to run) ---
# device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# model_check = AxCorCNN_Model(multiclass=False, mode='ax').to(device)
# axi_dummy = torch.randn(8, 40, 150, 120).to(device)
# coro_dummy = torch.randn(8, 32, 256, 256).to(device)
# va_logmar_dummy = torch.randn(8, 1).to(device)
#
# layer_outputs = {}
# def register_hook(module_name):
#     def hook(module, input, output):
#         layer_outputs[module_name] = output.shape
#     return hook
#
# for name, layer in model_check.named_modules():
#     if len(list(layer.children())) == 0:
#         layer.register_forward_hook(register_hook(name))
#
# with torch.no_grad():
#     model_check(axi_dummy, coro_dummy, va_logmar_dummy)
#
# for name, shape in layer_outputs.items():
#     print(f"{name}: {shape}")

# --- Example of calculating L1 norm for specific layers (uncomment to run) ---
# def sum_of_abs_weights_specific_layers(layers):
#     total_sum = 0.0
#     for layer in layers:
#         for name, param in layer.named_parameters():
#             if 'weight' in name and param.requires_grad:
#                 total_sum += torch.sum(torch.abs(param))
#     return total_sum.item()
#
# model_loaded = AxCorCNN_Model(multiclass=False, mode='ax')
# model_loaded.load_state_dict(torch.load('best_model.pth'))
#
# layers_to_sum_ax = [
#     model_loaded.ax_stem1, model_loaded.ax_stem2, model_loaded.ax_stem3, model_loaded.ax_stem4,
#     model_loaded.ax_conv, model_loaded.ax_dw
# ]
# layers_to_sum_cor = [
#     model_loaded.cor_stem1, model_loaded.cor_stem2, model_loaded.cor_stem3, model_loaded.cor_stem4,
#     model_loaded.cor_conv, model_loaded.cor_dw
# ]
#
# total_weight_abs_sum_ax = sum_of_abs_weights_specific_layers(layers_to_sum_ax)
# total_weight_abs_sum_cor = sum_of_abs_weights_specific_layers(layers_to_sum_cor)
# print(f"Sum of absolute values of specified weights for axial branch: {total_weight_abs_sum_ax}")
# print(f"Sum of absolute values of specified weights for coronal branch: {total_weight_abs_sum_cor}")


Number of batches in trainloader: 100
Number of batches in validationloader: 13
Total Trainable Params: 128729
Epoch: 1, Train Loss: 1.1199 (No Reg: 0.6940), Train Acc: 48.75% | Val Loss: 0.6905, Val Acc: 58.00%
  --> Saved best model weights at epoch 1 with validation accuracy 58.00%
Epoch: 2, Train Loss: 1.0951 (No Reg: 0.6918), Train Acc: 53.62% | Val Loss: 0.6913, Val Acc: 50.00%
Epoch: 3, Train Loss: 1.0763 (No Reg: 0.6926), Train Acc: 53.00% | Val Loss: 0.6930, Val Acc: 50.00%
Epoch: 4, Train Loss: 1.0551 (No Reg: 0.6879), Train Acc: 56.50% | Val Loss: 0.6850, Val Acc: 52.00%
Epoch: 5, Train Loss: 1.0377 (No Reg: 0.6845), Train Acc: 58.00% | Val Loss: 0.6763, Val Acc: 69.00%
  --> Saved best model weights at epoch 5 with validation accuracy 69.00%
Epoch: 6, Train Loss: 1.0216 (No Reg: 0.6796), Train Acc: 58.75% | Val Loss: 0.6729, Val Acc: 81.00%
  --> Saved best model weights at epoch 6 with validation accuracy 81.00%
Epoch: 7, Train Loss: 1.0074 (No Reg: 0.6742), Train Acc: 61.

In [27]:
plot_training_history(train_history)


In [28]:
fig = plot_confusion_matrix_plotly(conf_matrix, 0.5)


In [29]:
fig

In [30]:

# model = AxCorCNN_Model(multiclass=False , mode= 'ax')
# # total_params = count_parameters(model)
# # print(f'Total number of parameters: {total_params}')

# # def get_conv_layer(model, conv_layer_name):
# #     for name, layer in model.named_modules():
# #         if name == conv_layer_name:
# #             return layer
# #     raise ValueError(f"Layer '{conv_layer_name}' not found in the model.")

# trained_model = train(model, trainloader,validationloader,NUM_EPOCHS = 10 , lr =1e-4 , l1_lambda=1e-4, l2_lambda=1e-4)
# trained_model = AxCorCNN_Model(multiclass=False , mode= 'ax')
# trained_model.load_state_dict(torch.load('best_model.pth'))
# trained_model.eval()
# test(trained_model, validationloader)

In [31]:
# OLD HEATMAPS

# !rm -rf originals
# !rm -rf featuremaps
# !rm -rf fracture_heatmaps
# originals_dir = 'originals' # Not strictly needed if data is numpy
# os.makedirs('featuremaps', exist_ok=True) # Ensure this exists if images are saved there

# os.makedirs(originals_dir, exist_ok=True) # Ensure this exists if images are saved there
# os.makedirs(os.path.join(originals_dir, 'axial'), exist_ok=True)
# os.makedirs(os.path.join(originals_dir, 'coronal'), exist_ok=True)
# trained_model.to(device).eval()
# with torch.no_grad():


#     for batch_idx, data in enumerate(validationloader):

#         # print(feature_maps.shape)


#         # print(batch_idx)
#         axial , coronal ,VAlogmar, labels = data['axial'].type(torch.cuda.FloatTensor).to(device), data['coronal'].type(torch.cuda.FloatTensor).to(device) , data['VAlogmar'].type(torch.cuda.FloatTensor).to(device), data['label'].type(torch.cuda.FloatTensor).to(device)

#         # output = trained_model(input,VAlogmar)
#         outputs = trained_model(axial,coronal,VAlogmar)
#         # the class with the highest energy is what we choose as prediction
#         predictions = F.sigmoid(outputs).round()
#         #_, gt = torch.max(labels.data, 1)
#         # total += len(labels)#labels.size(0)
#         # correct += (predicted == labels).sum().item()

#         # for t, p in zip(labels.view(-1), predicted.view(-1)):
#             # conf_matrix[t.long(), p.long()] += 1

#         predicted_string = ["No Fracture" if x == 0 else "Fracture" for x in predictions]
#         label_string = ["No Fracture" if x == 0 else "Fracture" for x in labels]


#         # predictions = torch.round(torch.sigmoid(output))
#         # Iterate over the batch and process each sample
#         # for i in range(input.size(0)):  # data.size(0) is the batch size
#         #     sample = input[i]  # Single 16-channel image: [16, 128, 128]
#         #     label = labels[i]
#         #     prediction = predictions[i]

#         #     # Check if the prediction is correct or not and save accordingly
#         #     if prediction == 1 and label == 1:
#         #         # True Positive
#         #         save_as_numpy_and_video(sample, 'TP1', batch_idx * input.size(0) + i)
#         #     elif prediction == 0 and label == 0:
#         #         # True Negative
#         #         save_as_numpy_and_video(sample, 'TN1', batch_idx * input.size(0) + i)
#         #     elif prediction == 1 and label == 0:
#         #         # False Positive
#         #         save_as_numpy_and_video(sample, 'FP1', batch_idx * input.size(0) + i)
#         #     elif prediction == 0 and label == 1:
#         #         # False Negative
#         #         save_as_numpy_and_video(sample, 'FN1', batch_idx * input.size(0) + i)

#         for sample_idx in range(axial.size(0)):
#           coronal_image = coronal[sample_idx , 13:19 , :  , :].detach().cpu().numpy()
#           axial_image = axial [sample_idx  , 17:23 , :, :].detach().cpu().numpy()

#           label = labels[sample_idx]
#           prediction = predictions[sample_idx]
#           # original_image = sample[ 12:18 , : , :].detach().cpu().numpy()
#           plt.figure(figsize=(20, 2))
#           for idx in range(coronal_image.shape[0]):
#               plt.subplot(1, coronal_image.shape[0], idx + 1)
#               plt.imshow(coronal_image[idx], cmap='gray')
#               plt.axis('off')
#           plt.suptitle(f'coronal sample {sample_idx} , label = {label_string[sample_idx]},  predicted = {predicted_string[sample_idx]}')
#           plt.savefig(f'originals/coronal/coronal_sample_batch_{batch_idx}_sample_{sample_idx}.png', dpi=300, bbox_inches='tight')

#           plt.figure(figsize=(20, 2))
#           for idx in range(axial_image.shape[0]):
#               plt.subplot(1, axial_image.shape[0], idx + 1)
#               plt.imshow(axial_image[idx], cmap='gray')
#               plt.axis('off')
#           plt.suptitle(f'axial sample {sample_idx} , label = {label_string[sample_idx]},  predicted = {predicted_string[sample_idx]}')
#           plt.savefig(f'originals/axial/axial_sample_batch_{batch_idx}_sample_{sample_idx}.png', dpi=300, bbox_inches='tight')

#         batch_feature_maps = trained_model.feature_maps  # Get the feature maps

#         print(batch_feature_maps.size(0))


#         for sample_idx in range(batch_feature_maps.size(0)):

#           feature_maps= batch_feature_maps[sample_idx]

#           # print(feature_maps.shape)

#           feature_maps = feature_maps[ 5:11 , : , :].detach().cpu().numpy()
#           num_feature_maps = feature_maps.shape[0]
#           # print(num_feature_maps)
#           # # plt.figure(figsize=(20, 2))
#           for idx in range(num_feature_maps):
#               plt.subplot(1, num_feature_maps, idx + 1)
#               plt.imshow(feature_maps[idx], cmap='jet')
#               plt.axis('off')
#           plt.suptitle(f'sample {sample_idx} , label = {label_string[sample_idx]} , predicted = {predicted_string[sample_idx]}')
#           plt.savefig(f'featuremaps/feature_maps_sample_batch_{batch_idx}_sample_{sample_idx}.png', dpi=300, bbox_inches='tight')
#         # break

# # # Path to the folders to zi
# # !zip -r classification_results.zip TP1 TN1 FP1 FN1

In [32]:

# # --- Refactored Grad-CAM Generation Function ---

# def generate_grad_cam_heatmaps(
#     model: nn.Module,
#     dataloader: torch.utils.data.DataLoader,
#     target_conv_layer_name: str,
#     output_dir: str,
#     device: torch.device,
#     l1_lambda: float = 0.0001,
#     l2_lambda: float = 0.0001,
#     num_samples_to_process: int = None, # Limit processing for testing/debugging
#     slice_to_visualize: int = None # Specific slice index to pick from the 3D input for visualization
# ):
#     """
#     Generates and saves Grad-CAM heatmaps for samples from a DataLoader,
#     overlaying them on a chosen slice of the original image.

#     Args:
#         model (nn.Module): The trained PyTorch model.
#         dataloader (DataLoader): DataLoader for the data (e.g., validationloader).
#         target_conv_layer_name (str): The name of the convolutional layer to target for Grad-CAM.
#         output_dir (str): Directory to save the generated heatmap images.
#         device (torch.device): The device (CPU or CUDA) to run computations on.
#         l1_lambda (float): L1 regularization strength (used in loss calculation for gradients).
#         l2_lambda (float): L2 regularization strength (used in loss calculation for gradients).
#         num_samples_to_process (int, optional): If provided, processes only this many samples.
#                                                 Useful for quick testing. Defaults to None (process all).
#         slice_to_visualize (int, optional): If your input image is 3D (e.g., Slices, H, W),
#                                             specify which slice index to use for visualization.
#                                             If None, the middle slice will be chosen.
#     """
#     model.eval() # Set model to evaluation mode
#     model.to(device)
#     criterion = nn.BCEWithLogitsLoss()

#     # Ensure output directory exists and is clean if desired (uncomment next two lines for cleanup)
#     # shutil.rmtree(output_dir, ignore_errors=True)
#     os.makedirs(output_dir, exist_ok=True)

#     conv_layer = get_conv_layer(model, target_conv_layer_name)

#     sample_count = 0
#     with torch.enable_grad(): # Re-enable gradients for this block
#         for batch_idx, data in enumerate(dataloader):
#             if num_samples_to_process is not None and sample_count >= num_samples_to_process:
#                 break

#             # print(data.keys())
#             # break

#             # Assuming 'data' dict contains 'data' (image), 'VAlogmar', and 'label'
#             axial = data['axial'].type(torch.cuda.FloatTensor).to(device)
#             coronal = data['coronal'].type(torch.cuda.FloatTensor).to(device)
#             valogmar = data['VAlogmar'].type(torch.cuda.FloatTensor).to(device)
#             labels = data['label'].type(torch.cuda.FloatTensor).to(device)

#             # Define lists to store activations and gradients
#             activations = [None]
#             gradients = [None]

#             # Register hooks for the target convolutional layer
#             def forward_hook(module, input, output):
#                 activations[0] = output

#             def backward_hook(module, grad_input, grad_output):
#                 gradients[0] = grad_output[0]

#             hook_forward = conv_layer.register_forward_hook(forward_hook)
#             hook_backward = conv_layer.register_backward_hook(backward_hook)

#             for i in range(axial.size(0)): # Iterate through each sample in the batch
#                 if num_samples_to_process is not None and sample_count >= num_samples_to_process:
#                     break

#                 sample_input_axial = axial[i].unsqueeze(0) # Add batch dimension for single sample
#                 sample_input_coronal = coronal[i].unsqueeze(0) # Add batch dimension for single sample
#                 sample_valogmar = valogmar[i].unsqueeze(0)
#                 sample_label = labels[i].unsqueeze(0)

#                 # Ensure input tensors require gradients for Grad-CAM
#                 sample_input_axial.requires_grad_(True)
#                 sample_input_coronal.requires_grad_(True)
#                 sample_valogmar.requires_grad_(True) # If VAlogmar influences gradients significantly

#                 # Forward pass
#                 model_output = model(sample_input_axial , sample_input_coronal, sample_valogmar)
#                 predicted_prob = torch.sigmoid(model_output).item()
#                 print(predicted_prob)
#                 print(type(predicted_prob))

#                 predicted_label = np.round(predicted_prob) # 0 or 1

#                 # Prepare label strings for output filename and title
#                 true_label_str = "Fracture" if sample_label.item() == 1 else "No Fracture"
#                 predicted_label_str = "Fracture" if predicted_label == 1 else "No Fracture"

#                 # Calculate loss for backpropagation. For Grad-CAM, you typically backpropagate
#                 # a specific class score (e.g., the score of the predicted class).
#                 # Here, we backpropagate the overall BCEWithLogitsLoss as it was used during training.
#                 loss = criterion(model_output, sample_label)

#                 # Include regularization in the loss if that's how gradients were influenced during training.
#                 # If you want a "purer" saliency map for the prediction, you might exclude regularization here.
#                 l1_norm = sum(p.abs().sum() for p in model.parameters())
#                 l2_norm = sum(p.pow(2.0).sum() for p in model.parameters())
#                 loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm

#                 # Zero gradients before backward pass for correct accumulation
#                 model.zero_grad() # Clears gradients for model parameters
#                 if sample_input_axial.grad is not None: # Clear gradients for the input tensor
#                     sample_input_axial.grad.zero_()
#                 if sample_valogmar.grad is not None:
#                     sample_valogmar.grad.zero_()
#                 if sample_input_coronal.grad is not None:
#                     sample_input_coronal.grad.zero_()
#                 # Backward pass
#                 loss.backward()

#                 # Extract activations and gradients from the hooks
#                 # `squeeze(0)` removes the batch dimension added for the single sample
#                 feature_maps = activations[0].squeeze(0).cpu().detach().numpy() # (Channels, H_fm, W_fm)
#                 grads_val = gradients[0].squeeze(0).cpu().detach().numpy() # (Channels, H_fm, W_fm)

#                 # Global average pooling of gradients for each feature map
#                 pooled_grads = np.mean(grads_val, axis=(1, 2)) # Resulting shape (Channels,)

#                 # Multiply each feature map by its corresponding gradient weight
#                 for channel_idx, grad_weight in enumerate(pooled_grads):
#                     feature_maps[channel_idx, :, :] *= grad_weight

#                 # Sum along the channel dimension to get the raw heatmap
#                 heatmap = np.sum(feature_maps, axis=0) # Sum instead of mean, as per standard Grad-CAM
#                 heatmap = np.maximum(heatmap, 0) # Apply ReLU to the heatmap

#                 # Normalize heatmap to [0, 1] for proper visualization
#                 heatmap_min, heatmap_max = np.min(heatmap), np.max(heatmap)
#                 if heatmap_max - heatmap_min == 0:
#                     heatmap_normalized = np.zeros_like(heatmap) # Handle case where all values are the same
#                 else:
#                     heatmap_normalized = (heatmap - heatmap_min) / (heatmap_max - heatmap_min)

#                 # Get the original image slice to overlay on
#                 # Assuming `sample_input` is (1, Slices, H, W)
#                 original_img_np_axial = sample_input_axial.squeeze(0).detach().cpu().numpy() # (Slices, H, W)
#                 original_img_np_coronal = sample_input_coronal.squeeze(0).detach().cpu().numpy() # (Slices, H, W)




#                 # Determine which slice to visualize
#                 if original_img_np_axial.ndim == 3: # If it's a 3D volume (e.g., Slices, H, W)
#                     if slice_to_visualize is None:
#                         # Default to middle slice if not specified
#                         display_slice_idx = original_img_np_axial.shape[0] // 2
#                     else:
#                         display_slice_idx = slice_to_visualize
#                     img_to_overlay_axial = original_img_np_axial[display_slice_idx, :, :]
#                 elif original_img_np_axial.ndim == 2: # If it's already a 2D image (H, W)
#                     img_to_overlay_axial = original_img_np_axial
#                 else:
#                     print(f"Warning: Unexpected image dimensions {original_img_np_axial.shape} for visualization.")
#                     img_to_overlay_axial = original_img_np_axial[0, :, :] # Fallback to first channel/slice


#                 # Overlay heatmap using your provided function
#                 output_img = overlay_heatmap(img_to_overlay_axial, heatmap_normalized, alpha=0.5)





#                 # Determine which slice to visualize
#                 if original_img_np_coronal.ndim == 3: # If it's a 3D volume (e.g., Slices, H, W)
#                     if slice_to_visualize is None:
#                         # Default to middle slice if not specified
#                         display_slice_idx = original_img_np_coronal.shape[0] // 2
#                     else:
#                         display_slice_idx = slice_to_visualize
#                     img_to_overlay_coronal = original_img_np_coronal[display_slice_idx, :, :]
#                 elif original_img_np_coronal.ndim == 2: # If it's already a 2D image (H, W)
#                     img_to_overlay_coronal = original_img_np_coronal
#                 else:
#                     print(f"Warning: Unexpected image dimensions {original_img_np_coronal.shape} for visualization.")
#                     img_to_overlay_coronal = original_img_np_coronal[0, :, :] # Fallback to first channel/slice


#                 # Overlay heatmap using your provided function
#                 output_img_axial = overlay_heatmap(img_to_overlay_axial, heatmap_normalized, alpha=0.5)
#                 output_img_coronal = overlay_heatmap(img_to_overlay_coronal, heatmap_normalized, alpha=0.5)







#                 # Construct filename for saving
#                 filename = (
#                     f"batch_{batch_idx}_sample_{i}_"
#                     f"True_{true_label_str.replace(' ', '')}_"
#                     f"Pred_{predicted_label_str.replace(' ', '')}_"
#                     f"Prob_{predicted_prob:.2f}.jpg"
#                 )
#                 save_path = os.path.join(output_dir, filename)

#                 # Save the image using OpenCV
#                 cv2.imwrite(save_path, output_img)
#                 print(f"Saved Grad-CAM for batch {batch_idx}, sample {i} to {save_path}")

#                 sample_count += 1

#             # Remove hooks after processing each batch to prevent issues or memory leaks
#             hook_forward.remove()
#             hook_backward.remove()

#     print(f"\nFinished generating Grad-CAM heatmaps for {sample_count} samples.")
# trained_model = AxCorCNN_Model(multiclass=False, mode='ax')
# trained_model.load_state_dict(torch.load('best_model.pth'))
# trained_model.eval() # Important for inference and correct behavior of BatchNorm/Dropout

# device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# trained_model.to(device)

# l1_lambda = 1e-4 # From your snippet
# l2_lambda = 1e-4 # From your snippet

# # Define output directory for heatmaps
# gradcam_fracture_dir = 'gradcam_fracture'
# # shutil.rmtree(gradcam_output_dir, ignore_errors=True) # Uncomment if you want to clear on each run
# os.makedirs(gradcam_fracture_dir, exist_ok=True)

# # Call the new function
# # Set `num_samples_to_process` to None to process all, or a small number for testing.
# # `slice_to_visualize` can be adjusted based on the depth of your axial images.



# # --- keep your get_conv_layer(...) as-is (or this minimal one) ---
# def get_conv_layer(model: nn.Module, dotted: str) -> nn.Module:
#     m = model
#     for name in dotted.split('.'):
#         m = getattr(m, name)
#     return m

# # Resize-safe overlay
# def overlay_heatmap(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.5) -> np.ndarray:
#     # Ensure same HxW
#     if heatmap.shape != img.shape:
#         heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_CUBIC)

#     img_f = img.astype(np.float32)
#     if img_f.max() > img_f.min():
#         img_f = (img_f - img_f.min()) / (img_f.max() - img_f.min())
#     img_u8 = (img_f * 255).astype(np.uint8)

#     heat_u8 = (np.clip(heatmap, 0, 1) * 255).astype(np.uint8)
#     heat_color = cv2.applyColorMap(heat_u8, cv2.COLORMAP_JET)
#     img_color  = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)

#     return cv2.addWeighted(img_color, 1 - alpha, heat_color, alpha, 0)

# # --- Refactored Grad-CAM Generation Function ---
# def generate_grad_cam_heatmaps(
#     model: nn.Module,
#     dataloader: torch.utils.data.DataLoader,
#     target_conv_layer_name: str,
#     output_dir: str,
#     device: torch.device,
#     num_samples_to_process: int = None,
#     slice_to_visualize: int = None,
#     # which_class: int = None,  # None -> use predicted class; else 0 or 1
# ):
#     model.eval().to(device)
#     os.makedirs(output_dir, exist_ok=True)

#     conv_layer = get_conv_layer(model, target_conv_layer_name)

#     # Hooks (once per batch; we keep them active across samples in the batch)
#     activations = [None]
#     gradients = [None]
#     def fwd_hook(_, __, out): activations[0] = out
#     def bwd_hook(_, grad_in, grad_out): gradients[0] = grad_out[0]

#     sample_count = 0
#     with torch.enable_grad():
#         for batch_idx, data in enumerate(dataloader):
#             # if num_samples_to_process is not None and sample_count >= num_samples_to_process:
#                 # break

#             # ✅ device/dtype-safe moves
#             axial    = data['axial']   .to(device, dtype=torch.float32)
#             coronal  = data['coronal'] .to(device, dtype=torch.float32)
#             valogmar = data['VAlogmar'].to(device, dtype=torch.float32)
#             labels   = data['label']   .to(device, dtype=torch.float32)

#             # register hooks for this batch
#             h_f = conv_layer.register_forward_hook(fwd_hook)
#             h_b = conv_layer.register_full_backward_hook(bwd_hook)

#             # Forward whole batch (for predicted class)
#             logits = model(axial, coronal, valogmar).view(-1)       # [B]
#             probs  = torch.sigmoid(logits)
#             preds  = (probs >= 0.5).long()

#             B = axial.size(0)
#             for i in range(B):
#                 # if num_samples_to_process is not None and sample_count >= num_samples_to_process:
#                     # break

#                 # Single-sample views
#                 ax_i = axial[i:i+1].detach().clone()   # [1,S,H,W] or [1,H,W]
#                 co_i = coronal[i:i+1].detach().clone()
#                 va_i = valogmar[i:i+1].detach().clone()
#                 # print(ax_i.shape)
#                 # print(co_i.shape)
#                 # print(ax_i.shape)

#                 # Re-forward one sample to populate hooks cleanly
#                 logit_i = model(ax_i, co_i, va_i).view(-1)[0]       # scalar
#                 prob_i  = float(torch.sigmoid(logit_i).item())
#                 pred_i  = int(preds[i].item())

#                 # Grad-CAM target: class-1 score = logit; class-0 = -logit
#                 score = logit_i if pred_i == 1 else -logit_i

#                 model.zero_grad(set_to_none=True)
#                 score.backward()  # triggers hooks

#                 # Pull hooked tensors
#                 A  = activations[0]; dA = gradients[0]
#                 if A is None or dA is None:
#                     print("Warning: hooks didn't capture tensors; check target_conv_layer_name.")
#                     continue
#                 A  = A.detach().cpu().squeeze(0).numpy()     # [C,Hf,Wf]
#                 dA = dA.detach().cpu().squeeze(0).numpy()    # [C,Hf,Wf]

#                 # Weights via GAP over spatial dims
#                 w = dA.mean(axis=(1, 2))                     # [C]
#                 cam = (A * w[:, None, None]).sum(axis=0)     # [Hf,Wf]
#                 cam = np.maximum(cam, 0)
#                 cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

#                 # Pick slices
#                 ax_np = ax_i.detach().cpu().squeeze(0).numpy()  # [S,H,W] or [H,W]
#                 if ax_np.ndim == 3:
#                     S = ax_np.shape[0]
#                     sidx_ax = S//2 if slice_to_visualize is None else max(0, min(S-1, slice_to_visualize))
#                     img_ax = ax_np[sidx_ax]
#                 else:
#                     img_ax = ax_np

#                 co_np = co_i.detach().cpu().squeeze(0).numpy()
#                 if co_np.ndim == 3:
#                     S = co_np.shape[0]
#                     sidx_co = S//2 if slice_to_visualize is None else max(0, min(S-1, slice_to_visualize))
#                     img_co = co_np[sidx_co]
#                 else:
#                     img_co = co_np

#                 # print(img_ax.shape)
#                 # print(img_co.shape)
#                 # print(cam.shape)
#                 # print("-------------------------------")
#                 # break
#                 # Overlays (overlay will resize CAM to each slice size)
#                 out_ax = overlay_heatmap(img_ax, cam, alpha=0.5)
#                 out_co = overlay_heatmap(img_co, cam, alpha=0.5)
#                 # print(out_ax.shape)
#                 # print(out_co.shape)
#                 # print(out_ax[:, : , 0])
#                 # print("-------------------------------")
#                 # print(out_ax[:, : , 1])
#                 # Filenames
#                 true_i = int(labels[i].view(-1)[0].item())
#                 true_str = "Fracture" if true_i == 1 else "NoFracture"
#                 pred_str = "Fracture"  if pred_i  == 1 else "NoFracture"
#                 base = f"b{batch_idx}_samp{i}_True_{true_str}_Pred_{pred_str}_Prob_{prob_i:.2f}"

#                 cv2.imwrite(os.path.join(output_dir, base + "_AX.jpg"), out_ax)
#                 cv2.imwrite(os.path.join(output_dir, base + "_CO.jpg"), out_co)
#                 # print(f"Saved {base}_AX.jpg and {base}_CO.jpg")

#                 sample_count += 1

#             # remove hooks for this batch
#             h_f.remove(); h_b.remove()

#     print(f"\nFinished generating Grad-CAM heatmaps for {sample_count} samples.")


# generate_grad_cam_heatmaps(
#     model=trained_model,
#     dataloader=validationloader, # Assuming validationloader is defined and ready
#     target_conv_layer_name='ax_conv', # Or 'ax_stem4', or the last conv layer before flattening
#     output_dir=gradcam_fracture_dir,
#     device=device,
#     # l1_lambda=l1_lambda,
#     # l2_lambda=l2_lambda,
#     num_samples_to_process=10, # Example: Process only 10 samples for a quick check
#     slice_to_visualize=15 # Example: Visualize slice 15 (if your input is 3D and has >= 16 slices)
# )

In [33]:

!zip -r fracture_heatmap gradcam_fracture

  adding: gradcam_fracture/ (stored 0%)


In [34]:


def get_module_by_name(model: nn.Module, dotted: str) -> nn.Module:
    m = model
    for name in dotted.split('.'):  # supports nested modules like 'ax_branch.layer3'
        m = getattr(m, name)
    return m


def generate_branch_gradcam(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    ax_layer_name: str = 'ax_conv',
    cor_layer_name: str = 'cor_conv',
    output_dir: str = 'gradcam_fracture',
    device: torch.device = None,
    num_samples_to_process: int = None,
    slice_to_visualize: int = None,
):
    """
    Generate Grad-CAM heatmaps for BOTH branches independently by hooking
    an axial conv layer and a coronal conv layer in separate backward passes.

    Saves two images per sample: *_AX.jpg and *_CO.jpg.
    """
    os.makedirs(output_dir, exist_ok=True)

    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    model.eval().to(device)

    # Local hook storage reused per sample
    activations = [None]
    gradients = [None]

    def fwd_hook(_, __, out):
        activations[0] = out

    def bwd_hook(_, grad_in, grad_out):
        gradients[0] = grad_out[0]

    def compute_cam_for_layer(layer: nn.Module, ax_i, co_i, va_i, use_class_one: bool) -> np.ndarray:
        """Run a forward/backward with hooks on the given layer and return a normalized CAM."""
        # Clear old hook state
        activations[0] = None
        gradients[0] = None

        h_f = layer.register_forward_hook(fwd_hook)
        h_b = layer.register_full_backward_hook(bwd_hook)

        # Forward single sample to populate hooks
        logit = model(ax_i, co_i, va_i).view(-1)[0]
        score = logit if use_class_one else -logit

        model.zero_grad(set_to_none=True)
        score.backward(retain_graph=True)

        A = activations[0]
        dA = gradients[0]

        # Safety: if hooks didn't fire
        if A is None or dA is None:
            h_f.remove(); h_b.remove()
            return None

        A = A.detach().cpu().squeeze(0).numpy()   # [C,Hf,Wf]
        dA = dA.detach().cpu().squeeze(0).numpy() # [C,Hf,Wf]

        w = dA.mean(axis=(1, 2))                  # [C]
        cam = (A * w[:, None, None]).sum(axis=0)  # [Hf,Wf]
        cam = np.maximum(cam, 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        h_f.remove(); h_b.remove()
        return cam

    sample_count = 0
    with torch.enable_grad():
        for batch_idx, batch in enumerate(dataloader):
            if num_samples_to_process is not None and sample_count >= num_samples_to_process:
                break

            axial = batch['axial'].to(device, dtype=torch.float32)
            coronal = batch['coronal'].to(device, dtype=torch.float32)
            valogmar = batch['VAlogmar'].to(device, dtype=torch.float32)
            labels = batch['label'].to(device, dtype=torch.float32)

            B = axial.size(0)
            # Precompute predictions once per batch
            with torch.no_grad():
                logits_b = model(axial, coronal, valogmar).view(-1)
                probs_b = torch.sigmoid(logits_b)
                preds_b = (probs_b >= 0.5).long()

            for i in range(B):
                if num_samples_to_process is not None and sample_count >= num_samples_to_process:
                    break

                ax_i = axial[i:i+1].detach().clone()
                co_i = coronal[i:i+1].detach().clone()
                va_i = valogmar[i:i+1].detach().clone()

                prob_i = float(probs_b[i].item())
                pred_i = int(preds_b[i].item())
                true_i = int(labels[i].view(-1)[0].item())

                # Which class to target: use predicted class score
                use_class_one = (pred_i == 1)

                # Get modules
                ax_layer = get_module_by_name(model, ax_layer_name)
                cor_layer = get_module_by_name(model, cor_layer_name)

                # CAMs per branch (separate backward passes)
                cam_ax = compute_cam_for_layer(ax_layer, ax_i, co_i, va_i, use_class_one)
                cam_co = compute_cam_for_layer(cor_layer, ax_i, co_i, va_i, use_class_one)

                # If either CAM failed, skip
                if cam_ax is None or cam_co is None:
                    continue

                # Choose slices
                ax_np = ax_i.detach().cpu().squeeze(0).numpy()  # [S,H,W] or [H,W]
                if ax_np.ndim == 3:
                    S = ax_np.shape[0]
                    sidx_ax = S//2 if slice_to_visualize is None else max(0, min(S-1, slice_to_visualize))
                    img_ax = ax_np[sidx_ax]
                else:
                    img_ax = ax_np

                co_np = co_i.detach().cpu().squeeze(0).numpy()
                if co_np.ndim == 3:
                    S = co_np.shape[0]
                    sidx_co = S//2 if slice_to_visualize is None else max(0, min(S-1, slice_to_visualize))
                    img_co = co_np[sidx_co]
                else:
                    img_co = co_np

                # Overlays using existing overlay_heatmap(img, heatmap, alpha)
                out_ax = overlay_heatmap(img_ax, cam_ax, alpha=0.2)
                out_co = overlay_heatmap(img_co, cam_co, alpha=0.2)

                true_str = "Fracture" if true_i == 1 else "No Fracture"
                pred_str = "Fracture" if pred_i == 1 else "No Fracture"
                base = f"b{batch_idx}_s{i}_True_{true_str}_Pred_{pred_str}_Prob_{prob_i:.2f}"

                cv2.imwrite(os.path.join(output_dir, base + "_AX.jpg"), out_ax)
                cv2.imwrite(os.path.join(output_dir, base + "_CO.jpg"), out_co)

                sample_count += 1

    print(f"Finished generating Grad-CAM heatmaps for {sample_count} samples.")


In [35]:
# Run dual-branch Grad-CAM
output_dir = 'gradcam_fracture'
os.makedirs(output_dir, exist_ok=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
trained_model.eval().to(device)

# Adjust layer names if your model's attributes differ
generate_branch_gradcam(
    model=trained_model,
    dataloader=validationloader,
    ax_layer_name='ax_conv',
    cor_layer_name='cor_conv',
    output_dir='test',
    device=device,
    num_samples_to_process=10,
    slice_to_visualize=15,
)


Finished generating Grad-CAM heatmaps for 10 samples.


In [36]:

def _normalize01(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    mn, mx = float(a.min()), float(a.max())
    if mx > mn:
        return (a - mn) / (mx - mn)
    return np.zeros_like(a, dtype=np.float32)

def generate_branch_gradcam_plotly(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    ax_layer_name: str = 'ax_conv',     # use layers that actually run in forward()
    cor_layer_name: str = 'cor_conv',
    device: torch.device | None = None,
    output_dir: str = 'gradcam_plotly',
    num_samples_to_process: int | None = 10,
    slice_to_visualize: int | None = None,
    overlay_alpha: float = 0.7,
    fig_width=1100,
    fig_height=800,
    hspace=0.06,
    wspace=0.03,
):
    """
    For each sample, creates an interactive HTML with 4 panels:
      (1,1) Axial original
      (1,2) Coronal original
      (2,1) Axial + axial-branch CAM
      (2,2) Coronal + coronal-branch CAM
    """
    os.makedirs(output_dir, exist_ok=True)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.eval().to(device)

    ax_layer = get_module_by_name(model, ax_layer_name)
    co_layer = get_module_by_name(model, cor_layer_name)

    # buffers
    activations = [None]
    gradients = [None]
    def fwd_hook(_, __, out): activations[0] = out
    def bwd_hook(_, grad_in, grad_out): gradients[0] = grad_out[0]

    def compute_cam(layer: nn.Module, ax_i, co_i, va_i, use_class_one: bool) -> np.ndarray | None:
        activations[0] = gradients[0] = None
        h_f = layer.register_forward_hook(fwd_hook)
        h_b = layer.register_full_backward_hook(bwd_hook)

        logit = model(ax_i, co_i, va_i).view(-1)[0]
        score = logit if use_class_one else -logit
        model.zero_grad(set_to_none=True)
        score.backward(retain_graph=False)

        A, dA = activations[0], gradients[0]
        h_f.remove(); h_b.remove()
        if A is None or dA is None:
            return None
        A  = A.detach().cpu().squeeze(0).numpy()    # [C,Hf,Wf]
        dA = dA.detach().cpu().squeeze(0).numpy()   # [C,Hf,Wf]
        w  = dA.mean(axis=(1, 2))                   # [C]
        cam = (A * w[:, None, None]).sum(axis=0)    # [Hf,Wf]
        cam = np.maximum(cam, 0)
        cam = _normalize01(cam)
        return cam

    saved = 0
    with torch.enable_grad():
        for bidx, batch in enumerate(dataloader):
            if num_samples_to_process is not None and saved >= num_samples_to_process:
                break

            axial   = batch['axial']   .to(device, dtype=torch.float32)
            coronal = batch['coronal'] .to(device, dtype=torch.float32)
            va      = batch['VAlogmar'].to(device, dtype=torch.float32)
            labels  = batch['label']   .to(device, dtype=torch.float32)

            with torch.no_grad():
                logits = model(axial, coronal, va).view(-1)
                probs  = torch.sigmoid(logits)
                preds  = (probs >= 0.5).long()

            B = axial.size(0)
            for i in range(B):
                if num_samples_to_process is not None and saved >= num_samples_to_process:
                    break

                ax_i = axial[i:i+1].detach().clone()
                co_i = coronal[i:i+1].detach().clone()
                va_i = va[i:i+1].detach().clone()

                prob_i = float(probs[i].item())
                pred_i = int(preds[i].item())
                true_i = int(labels[i].view(-1)[0].item())
                use_class_one = (pred_i == 1)

                cam_ax = compute_cam(ax_layer, ax_i, co_i, va_i, use_class_one)
                cam_co = compute_cam(co_layer, ax_i, co_i, va_i, use_class_one)
                if cam_ax is None or cam_co is None:
                    continue

                ax_slice = _pick_slice(ax_i, slice_idx=slice_to_visualize)  # [H,W]
                co_slice = _pick_slice(co_i, slice_idx=slice_to_visualize)

                base_ax = _normalize01(ax_slice)
                base_co = _normalize01(co_slice)
                cam_ax  = _ensure_same_hw(cam_ax, base_ax)
                cam_co  = _ensure_same_hw(cam_co, base_co)

                # Build interactive figure
                fig = make_subplots(
                    rows=2, cols=2,
                    subplot_titles=("Axial", "Coronal", "Axial Grad-CAM", "Coronal Grad-CAM"),
                    # horizontal_spacing=0.03, vertical_spacing=0.06
                            horizontal_spacing=wspace,
        vertical_spacing=hspace
                )

                # Top row: originals
                _add_img_subplot(fig, 1, 1, base_ax, "Axial (original)")
                _add_img_subplot(fig, 1, 2, base_co, "Coronal (original)")

                # Bottom row: overlays (colorbar only on the last to declutter)
                _add_overlay_subplot(fig, 2, 1, base_ax, cam_ax, "Axial Grad-CAM", alpha=overlay_alpha, show_colorbar=False)
                _add_overlay_subplot(fig, 2, 2, base_co, cam_co, "Coronal Grad-CAM", alpha=overlay_alpha, show_colorbar=True)

                # Toggle buttons to hide/show CAM layers (the 2nd trace in each bottom subplot)
                # Trace order: [orig_ax, orig_co, base_ax, cam_ax, base_co, cam_co]
                fig.update_layout(
                            width=fig_width,
        height=fig_height,
                    title=f"True: {'Fracture' if true_i==1 else 'No Fracture'} | Pred: {'Fracture' if pred_i==1 else 'No Fracture'} | Prob(Fracture): {prob_i:.2f}",
                    margin=dict(l=10, r=10, t=50, b=10),
                    updatemenus=[dict(
                        type="buttons",
                        x=1.02, y=1.02, xanchor='left',
                        buttons=[
                            dict(
                                label="Hide overlays",
                                method="update",
                                args=[{"visible": [True, True, True, False, True, False]}]
                            ),
                            dict(
                                label="Show overlays",
                                method="update",
                                args=[{"visible": [True, True, True, True, True, True]}]
                            ),
                        ]
                    )],



                )

                # Save per-sample HTML
                true_str = "Fracture" if true_i == 1 else "No Fracture"
                pred_str = "SFractureevere" if pred_i == 1 else "No Fracture"
                fname = f"b{bidx}_i{i}_T{true_str}_P{pred_str}_Pr{prob_i:.2f}.html"
                pio.write_html(fig, file=os.path.join(output_dir, fname), auto_open=False, include_plotlyjs='cdn')
                saved += 1

    print(f"Saved {saved} interactive Plotly Grad-CAM files to '{output_dir}'.")


# ---------- main ----------
def generate_branch_gradcam(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    ax_layer_name: str = 'ax_conv',   # REPLACE with your actual axial conv name
    cor_layer_name: str = 'cor_conv', # REPLACE with your actual coronal conv name
    output_dir: str = 'gradcam_fracture',
    device: torch.device | None = None,
    num_samples_to_process: int | None = None,
    slice_to_visualize: int | None = None,
    alpha: float = 0.6,
    cmap: int = cv2.COLORMAP_TURBO,
):
    """
    For each sample:
      TL: axial original
      TR: coronal original
      BL: axial + axial-branch CAM
      BR: coronal + coronal-branch CAM
    Saved as a single composite image.
    """
    os.makedirs(output_dir, exist_ok=True)
    if device is None:
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.eval().to(device)

    # Resolve target layers once
    ax_layer = get_module_by_name(model, ax_layer_name)
    co_layer = get_module_by_name(model, cor_layer_name)

    # reusable hook buffers
    activations = [None]
    gradients = [None]

    def fwd_hook(_, __, out): activations[0] = out
    def bwd_hook(_, grad_in, grad_out): gradients[0] = grad_out[0]


    saved = 0
    with torch.enable_grad():
        for bidx, batch in enumerate(dataloader):
            # if num_samples_to_process is not None and saved >= num_samples_to_process:
                # break

            axial   = batch['axial']   .to(device, dtype=torch.float32)
            coronal = batch['coronal'] .to(device, dtype=torch.float32)
            va      = batch['VAlogmar'].to(device, dtype=torch.float32)
            labels  = batch['label']   .to(device, dtype=torch.float32)

            # one forward for predictions (no hooks needed)
            with torch.no_grad():
                logits = model(axial, coronal, va).view(-1)
                probs  = torch.sigmoid(logits)
                preds  = (probs >= 0.5).long()

            B = axial.size(0)
            for i in range(B):
                # if num_samples_to_process is not None and saved >= num_samples_to_process:
                    # break

                ax_i = axial[i:i+1].detach().clone()
                co_i = coronal[i:i+1].detach().clone()
                va_i = va[i:i+1].detach().clone()

                prob_i = float(probs[i].item())
                pred_i = int(preds[i].item())
                true_i = int(labels[i].view(-1)[0].item())
                use_class_one = (pred_i == 1)

                # compute branch-specific CAMs
                cam_ax = compute_cam(ax_layer, ax_i, co_i, va_i, use_class_one)
                cam_co = compute_cam(co_layer, ax_i, co_i, va_i, use_class_one)
                if cam_ax is None or cam_co is None:
                    # print("continueing")
                    continue

                # pick slices
                def pick_slice(vol: torch.Tensor) -> np.ndarray:
                    v = vol.squeeze(0).detach().cpu().numpy()
                    if v.ndim == 3:
                        S = v.shape[0]
                        idx = S//2 if slice_to_visualize is None else max(0, min(S-1, slice_to_visualize))
                        return v[idx]
                    return v

                ax_slice = pick_slice(ax_i)  # [H,W]
                co_slice = pick_slice(co_i)

                # build four panels (all BGR)
                ax_orig = cv2.cvtColor(_normalize_gray(ax_slice), cv2.COLOR_GRAY2BGR)
                co_orig = cv2.cvtColor(_normalize_gray(co_slice), cv2.COLOR_GRAY2BGR)
                ax_ol   = overlay_heatmap(ax_slice, cam_ax, alpha=alpha, colormap=cmap)
                co_ol   = overlay_heatmap(co_slice, cam_co, alpha=alpha, colormap=cmap)

                # add per-panel titles
                ax_orig = _put_title(ax_orig, "Axial (original)")
                co_orig = _put_title(co_orig, "Coronal (original)")
                ax_ol   = _put_title(ax_ol,   "Axial Grad-CAM")
                co_ol   = _put_title(co_ol,   "Coronal Grad-CAM")

                # print(ax_ol.shape)

                # header
                true_str = "Fracture" if true_i == 1 else "No Fracture"
                pred_str = "Fracture" if pred_i == 1 else "No Fracture"
                header = f"True: {true_str} | Pred: {pred_str} | Prob(Fracture): {prob_i:.2f}"

                # compose 2x2 and save
                grid = _compose_2x2(ax_orig, co_orig, ax_ol, co_ol, header=header)
                fname = f"b{bidx}_i{i}_T{true_str}_P{pred_str}_Pr{prob_i:.2f}.jpg"
                cv2.imwrite(os.path.join(output_dir, fname), grid)
                saved += 1

    print(f"Saved {saved} composite Grad-CAM images to '{output_dir}'.")


# generate_branch_gradcam(
#     model=trained_model,
#     dataloader=validationloader,
#     ax_layer_name='ax_stem4',   # ← put your actual axial last-conv here
#     cor_layer_name='cor_stem4', # ← and your actual coronal last-conv here
#     output_dir='gradcam_dual_composites',
#     device=device,
#     num_samples_to_process=20,
#     slice_to_visualize=15,      # or None for middle slice
#     alpha=0.65,                 # stronger color overlay
#     cmap=cv2.COLORMAP_TURBO     # vivid colormap
# )

generate_branch_gradcam_plotly(
    model=trained_model,
    dataloader=validationloader,
    ax_layer_name='ax_conv',     # ✔ used in forward
    cor_layer_name='cor_conv',   # ✔ used in forward
    output_dir='fracture_heatmaps',
    device=device,
    num_samples_to_process=60,
    slice_to_visualize=18,
    # alpha=0.65,
    # cmap=cv2.COLORMAP_TURBO
)

# Label encoding used throughout the paper: severe = 1 (positive), mild = 0 (negative)



Saved 60 interactive Plotly Grad-CAM files to 'fracture_heatmaps'.


In [37]:
!zip -r fracture_heatmaps.zip fracture_heatmaps

  adding: fracture_heatmaps/ (stored 0%)
  adding: fracture_heatmaps/b3_i1_TFracture_PSFractureevere_Pr0.54.html (deflated 61%)
  adding: fracture_heatmaps/b7_i0_TFracture_PSFractureevere_Pr0.64.html (deflated 61%)
  adding: fracture_heatmaps/b1_i7_TFracture_PSFractureevere_Pr0.64.html (deflated 62%)
  adding: fracture_heatmaps/b5_i3_TNo Fracture_PNo Fracture_Pr0.42.html (deflated 66%)
  adding: fracture_heatmaps/b0_i5_TFracture_PNo Fracture_Pr0.43.html (deflated 67%)
  adding: fracture_heatmaps/b5_i5_TFracture_PSFractureevere_Pr0.51.html (deflated 61%)
  adding: fracture_heatmaps/b6_i0_TNo Fracture_PSFractureevere_Pr0.56.html (deflated 62%)
  adding: fracture_heatmaps/b6_i1_TFracture_PSFractureevere_Pr0.61.html (deflated 62%)
  adding: fracture_heatmaps/b6_i3_TNo Fracture_PNo Fracture_Pr0.50.html (deflated 63%)
  adding: fracture_heatmaps/b0_i2_TNo Fracture_PSFractureevere_Pr0.57.html (deflated 61%)
  adding: fracture_heatmaps/b5_i1_TNo Fracture_PNo Fracture_Pr0.45.html (deflated 66%)

In [38]:
# --- Multi-model evaluation: Accuracy / Sensitivity / Specificity / F1 / AUROC (95% CI) ---
# This block trains and evaluates multiple backbones using the same dataloaders.


from roc_curve_utils import compute_binary_metrics
from fracture_model_zoo import make_model

def _collect_scores(model: nn.Module, dataloader):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model.eval().to(device)
    y_true = []
    y_score = []
    with torch.no_grad():
        for batch in dataloader:
            axial = batch['axial'].to(device, dtype=torch.float32)
            coronal = batch['coronal'].to(device, dtype=torch.float32)
            va = batch['VAlogmar'].to(device, dtype=torch.float32)
            labels = batch['label'].to(device, dtype=torch.float32).view(-1)
            logits = model(axial, coronal, va).view(-1)
            probs = torch.sigmoid(logits)
            y_true.append(labels.detach().cpu().numpy())
            y_score.append(probs.detach().cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_score)

def train_model_generic(
    model: nn.Module,
    trainloader,
    validationloader,
    num_epochs: int = 20,
    lr: float = 1e-4,
    l1_lambda: float = 1e-4,
    l2_lambda: float = 1e-4,
    save_path: str = 'best_model_tmp.pth',
):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(params=model.parameters(), lr=lr)
    best_val_acc = -1.0
    best_wts = None
    for epoch in range(num_epochs):
        model.train()
        for batch in trainloader:
            axial = batch['axial'].to(device, dtype=torch.float32)
            coronal = batch['coronal'].to(device, dtype=torch.float32)
            va = batch['VAlogmar'].to(device, dtype=torch.float32)
            labels = batch['label'].to(device, dtype=torch.float32)
            optimizer.zero_grad(set_to_none=True)
            logits = model(axial, coronal, va)
            loss_wo = criterion(logits, labels)
            l1 = sum(p.abs().sum() for p in model.parameters() if p.requires_grad)
            l2 = sum(p.pow(2.0).sum() for p in model.parameters() if p.requires_grad)
            loss = loss_wo + l1_lambda * l1 + l2_lambda * l2
            loss.backward()
            optimizer.step()
        yv, sv = _collect_scores(model, validationloader)
        predv = (sv >= 0.5).astype(int)
        acc = float((predv == yv.astype(int)).mean())
        if acc > best_val_acc:
            best_val_acc = acc
            best_wts = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(best_wts, save_path)
        print(f"Epoch {epoch+1:02d}/{num_epochs}: val acc={acc*100:.2f}% (best={best_val_acc*100:.2f}%)")
    if best_wts is not None:
        model.load_state_dict(best_wts)
    return model.cpu()

def evaluate_model_with_roc(
    model_name: str,
    trainloader,
    validationloader,
    axial_in_channels: int,
    coronal_in_channels: int,
    num_epochs: int = 20,
    lr: float = 1e-4,
    l1_lambda: float = 1e-4,
    l2_lambda: float = 1e-4,
    pretrained: bool = True,
    n_bootstrap: int = 2000,
    seed: int = 42,
):
    model = make_model(
        model_name=model_name,
        axial_in_channels=axial_in_channels,
        coronal_in_channels=coronal_in_channels,
        pretrained=pretrained,
    )
    save_path = f"best_{model_name.replace(' ', '_')}.pth"
    model = train_model_generic(
        model=model,
        trainloader=trainloader,
        validationloader=validationloader,
        num_epochs=num_epochs,
        lr=lr,
        l1_lambda=l1_lambda,
        l2_lambda=l2_lambda,
        save_path=save_path,
    )
    y_true, y_score = _collect_scores(model, validationloader)
    m = compute_binary_metrics(
        y_true=y_true,
        y_score=y_score,
        threshold=0.5,
        n_bootstrap=n_bootstrap,
        seed=seed,
    )
    return {
        'Model': model_name,
        'Accuracy': m.accuracy,
        'Sensitivity': m.sensitivity,
        'Specificity': m.specificity,
        'F1': m.f1,
        'AUROC': m.auroc,
        'AUROC 95% CI': f"[{m.auroc_ci95[0]:.3f}, {m.auroc_ci95[1]:.3f}]",
    }

batch0 = next(iter(trainloader))
axial_in_channels = int(batch0['axial'].shape[1])
coronal_in_channels = int(batch0['coronal'].shape[1])
print('axial_in_channels:', axial_in_channels, '| coronal_in_channels:', coronal_in_channels)

models_to_run = ['FractureDetectionCNN', 'EfficientNet', 'GoogLeNet', 'VGG16', 'ResNet18']
rows = []
for mn in models_to_run:
    print('\n===', mn, '===')
    rows.append(evaluate_model_with_roc(
        model_name=mn,
        trainloader=trainloader,
        validationloader=validationloader,
        axial_in_channels=axial_in_channels,
        coronal_in_channels=coronal_in_channels,
        num_epochs=4,
        lr=1e-4,
        l1_lambda=1e-4,
        l2_lambda=1e-4,
        pretrained=True,
        n_bootstrap=2000,
        seed=42,
    ))
df_results = pd.DataFrame(rows)
display(df_results)
df_results.to_csv('fracture_model_comparison_metrics.csv', index=False)


axial_in_channels: 40 | coronal_in_channels: 32

=== FractureDetectionCNN ===
Epoch 01/20: val acc=50.00% (best=50.00%)
Epoch 02/20: val acc=48.00% (best=50.00%)
Epoch 03/20: val acc=52.00% (best=52.00%)
Epoch 04/20: val acc=63.00% (best=63.00%)
Epoch 05/20: val acc=50.00% (best=63.00%)
Epoch 06/20: val acc=75.00% (best=75.00%)
Epoch 07/20: val acc=59.00% (best=75.00%)
Epoch 08/20: val acc=68.00% (best=75.00%)
Epoch 09/20: val acc=59.00% (best=75.00%)
Epoch 10/20: val acc=59.00% (best=75.00%)
Epoch 11/20: val acc=68.00% (best=75.00%)
Epoch 12/20: val acc=74.00% (best=75.00%)
Epoch 13/20: val acc=68.00% (best=75.00%)
Epoch 14/20: val acc=64.00% (best=75.00%)
Epoch 15/20: val acc=66.00% (best=75.00%)
Epoch 16/20: val acc=74.00% (best=75.00%)
Epoch 17/20: val acc=76.00% (best=76.00%)
Epoch 18/20: val acc=73.00% (best=76.00%)
Epoch 19/20: val acc=66.00% (best=76.00%)
Epoch 20/20: val acc=69.00% (best=76.00%)

=== EfficientNet ===
Downloading: "https://download.pytorch.org/models/efficientn

100%|██████████| 20.5M/20.5M [00:00<00:00, 215MB/s]


Epoch 01/20: val acc=70.00% (best=70.00%)
Epoch 02/20: val acc=67.00% (best=70.00%)
Epoch 03/20: val acc=68.00% (best=70.00%)
Epoch 04/20: val acc=66.00% (best=70.00%)
Epoch 05/20: val acc=67.00% (best=70.00%)
Epoch 06/20: val acc=60.00% (best=70.00%)
Epoch 07/20: val acc=60.00% (best=70.00%)
Epoch 08/20: val acc=68.00% (best=70.00%)
Epoch 09/20: val acc=65.00% (best=70.00%)
Epoch 10/20: val acc=65.00% (best=70.00%)
Epoch 11/20: val acc=66.00% (best=70.00%)
Epoch 12/20: val acc=68.00% (best=70.00%)
Epoch 13/20: val acc=66.00% (best=70.00%)
Epoch 14/20: val acc=67.00% (best=70.00%)
Epoch 15/20: val acc=67.00% (best=70.00%)
Epoch 16/20: val acc=64.00% (best=70.00%)
Epoch 17/20: val acc=65.00% (best=70.00%)
Epoch 18/20: val acc=68.00% (best=70.00%)
Epoch 19/20: val acc=64.00% (best=70.00%)
Epoch 20/20: val acc=68.00% (best=70.00%)

=== GoogLeNet ===


ValueError: The parameter 'aux_logits' expected value True but got False instead.